# Prediccion de Perdida de Clientes (Churn) - Olist

Analisis completo de prediccion de churn usando datos del marketplace Olist.
Se entrenan y evaluan multiples modelos de clasificacion, se aplica PCA para reduccion
de dimensionalidad y K-Means para segmentacion de clientes.

## 1. Configuracion del entorno

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, roc_curve, auc, classification_report

from src.data.load_data import load_csv
from src.data.preprocess import build_churn_dataset
from src.features.feature_engineering import create_churn_features, apply_pca
from src.data.split import split_dataset
from src.models.train import train_and_validate
from src.evaluation.evaluate import evaluate_all_models, get_feature_importance
from src.config.settings import RESULTS_DIR

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)

## 2. Analisis Exploratorio de Datos (EDA)

Carga y exploracion de los datasets raw de Olist para entender la estructura y distribucion de los datos.

In [ ]:
customers = load_csv("olist_customers_dataset.csv")
orders = load_csv("olist_orders_dataset.csv")
order_items = load_csv("olist_order_items_dataset.csv")
payments = load_csv("olist_order_payments_dataset.csv")
reviews = load_csv("olist_order_reviews_dataset.csv")

print("Dimensiones de los datasets:")
for name, df in [("Customers", customers), ("Orders", orders),
                  ("Order Items", order_items), ("Payments", payments),
                  ("Reviews", reviews)]:
    print(f"  {name}: {df.shape[0]} filas x {df.shape[1]} columnas")

In [ ]:
customers.head()

In [ ]:
print("Distribucion de estados de ordenes:")
print(orders["order_status"].value_counts())
print(f"\nOrdenes entregadas: {(orders['order_status'] == 'delivered').sum()} "
      f"({(orders['order_status'] == 'delivered').mean():.1%})")

In [ ]:
orders_temp = orders.copy()
orders_temp["order_purchase_timestamp"] = pd.to_datetime(orders_temp["order_purchase_timestamp"])

fig, ax = plt.subplots(figsize=(12, 4))
orders_temp.set_index("order_purchase_timestamp").resample("M").size().plot(ax=ax)
ax.set_title("Volumen de ordenes por mes")
ax.set_ylabel("Cantidad de ordenes")
ax.set_xlabel("Fecha")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

reviews["review_score"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Distribucion de review scores")
axes[0].set_ylabel("Cantidad")

customers["customer_state"].value_counts().head(10).plot(kind="bar", ax=axes[1], color="teal")
axes[1].set_title("Top 10 estados con mas clientes")
axes[1].set_ylabel("Cantidad")

plt.tight_layout()
plt.show()

## 3. Construccion del Dataset de Churn

Se construye un dataset a nivel de cliente unico, mergeando todas las tablas raw y creando la etiqueta de churn.
Un cliente se considera **churn** si no ha realizado compras en los ultimos **180 dias** respecto a la fecha de referencia del dataset.

In [ ]:
churn_data = build_churn_dataset()
print(f"Dataset de churn: {churn_data.shape[0]} clientes x {churn_data.shape[1]} columnas")
print(f"\nColumnas: {list(churn_data.columns)}")
churn_data.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_counts = churn_data["churn"].value_counts()
churn_counts.plot(kind="bar", ax=axes[0], color=["steelblue", "coral"])
axes[0].set_title("Distribucion de Churn")
axes[0].set_xticklabels(["No Churn (0)", "Churn (1)"], rotation=0)
axes[0].set_ylabel("Cantidad de clientes")

churn_counts.plot(kind="pie", ax=axes[1], autopct="%.1f%%", colors=["steelblue", "coral"])
axes[1].set_title("Proporcion de Churn")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

print(f"Clientes sin churn: {churn_counts.get(0, 0)}")
print(f"Clientes con churn: {churn_counts.get(1, 0)}")
print(f"Tasa de churn: {churn_data['churn'].mean():.2%}")

In [ ]:
numeric_cols = churn_data.select_dtypes(include="number").columns.drop("churn")
n_cols = 4
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3 * n_rows))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    churn_data[churn_data["churn"] == 0][col].hist(bins=30, ax=axes[idx], alpha=0.6, label="No Churn", color="steelblue")
    churn_data[churn_data["churn"] == 1][col].hist(bins=30, ax=axes[idx], alpha=0.6, label="Churn", color="coral")
    axes[idx].set_title(col, fontsize=9)
    axes[idx].legend(fontsize=7)

for idx in range(len(numeric_cols), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle("Distribucion de features por clase de churn", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
churn_data.describe()

## 4. Feature Engineering

Se transforman las features para el modelo:
- Escalado de variables numericas con StandardScaler
- Codificacion de variables categoricas con OneHotEncoder

In [ ]:
features, target, feature_names = create_churn_features(churn_data)
print(f"Features: {features.shape[1]} variables")
features.head()

In [ ]:
# correlacion de features con la variable objetivo
all_data = features.copy()
all_data["churn"] = target.values

correlations = all_data.corr()["churn"].drop("churn").abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
correlations.head(20).plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 20 features con mayor correlacion con churn")
ax.set_xlabel("Correlacion absoluta")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Entrenamiento de Modelos

### 5.1 Division de datos

In [ ]:
X_train, X_test, y_train, y_test = split_dataset(features, target)
print(f"Train set: {X_train.shape[0]} muestras ({X_train.shape[0]/len(features):.0%})")
print(f"Test set:  {X_test.shape[0]} muestras ({X_test.shape[0]/len(features):.0%})")
print(f"\nDistribucion de churn en train: {y_train.mean():.2%}")
print(f"Distribucion de churn en test:  {y_test.mean():.2%}")

### 5.2 Entrenamiento con Cross-Validation (StratifiedKFold, 5 folds)

Se entrenan 7 modelos de clasificacion. Cada modelo se valida con StratifiedKFold (5 folds) para obtener
estimaciones robustas del rendimiento y controlar el sobreajuste.

**Modelos:** Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, SVM, KNN, Naive Bayes

In [ ]:
trained_models, cv_results = train_and_validate(X_train, y_train, save_models=False)

In [ ]:
cv_display = cv_results[["cv_accuracy_mean", "cv_precision_mean", "cv_recall_mean", "cv_f1_mean"]].round(4)
cv_display.columns = ["Accuracy", "Precision", "Recall", "F1"]
print("Resultados de Cross-Validation:")
cv_display

In [ ]:
metrics_list = ["accuracy", "precision", "recall", "f1"]
models_list = cv_results.index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, metric in enumerate(metrics_list):
    means = cv_results[f"cv_{metric}_mean"]
    stds = cv_results[f"cv_{metric}_std"]
    axes[idx].barh(models_list, means, xerr=stds, capsize=5, color="steelblue", alpha=0.8)
    axes[idx].set_title(f"Cross-Validation {metric.capitalize()}")
    axes[idx].set_xlabel(metric.capitalize())
    axes[idx].set_xlim(0, 1)

plt.suptitle("Resultados de Cross-Validation (5-Fold)", fontsize=14)
plt.tight_layout()
plt.show()

### 5.3 Control de Sobreajuste

Comparamos el rendimiento en entrenamiento vs validacion (cross-validation) para detectar sobreajuste.
Si el score de entrenamiento es significativamente mayor que el de validacion, el modelo esta sobreajustando.

In [ ]:
overfit_data = pd.DataFrame({
    "Train F1": cv_results["train_f1_mean"],
    "CV F1": cv_results["cv_f1_mean"],
    "Gap": cv_results["train_f1_mean"] - cv_results["cv_f1_mean"]
}).round(4)

print("Analisis de sobreajuste (Train vs CV):")
print(overfit_data)

fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(models_list))
width = 0.35
ax.bar([i - width/2 for i in x], overfit_data["Train F1"], width, label="Train F1", color="steelblue")
ax.bar([i + width/2 for i in x], overfit_data["CV F1"], width, label="CV F1", color="coral")
ax.set_xticks(list(x))
ax.set_xticklabels(models_list, rotation=45, ha="right")
ax.set_ylabel("F1 Score")
ax.set_title("Train vs Cross-Validation F1 - Control de Sobreajuste")
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

print("\nInterpretacion:")
for model_name in models_list:
    gap = overfit_data.loc[model_name, "Gap"]
    if gap > 0.1:
        print(f"  {model_name}: GAP={gap:.4f} - SOBREAJUSTE detectado")
    elif gap > 0.05:
        print(f"  {model_name}: GAP={gap:.4f} - sobreajuste moderado")
    else:
        print(f"  {model_name}: GAP={gap:.4f} - bien generalizado")

## 6. Evaluacion en Test Set

Evaluamos los modelos entrenados en el conjunto de test (datos no vistos durante el entrenamiento).

In [ ]:
metrics_df, confusion_matrices, reports, predictions, probabilities = evaluate_all_models(
    trained_models, X_test, y_test,
    output_csv=os.path.join(RESULTS_DIR, "churn_metrics.csv")
)
metrics_df

In [ ]:
# metricas por modelo
df_plot = metrics_df.reset_index().rename(columns={"index": "Model"})

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, metric in enumerate(metrics_list):
    sns.barplot(data=df_plot, x="Model", y=metric, ax=axes[idx], color="steelblue")
    axes[idx].set_title(f"{metric.capitalize()} por modelo")
    axes[idx].set_ylabel(metric.capitalize())
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=45, ha="right")
    axes[idx].set_ylim(0, 1)
    for container in axes[idx].containers:
        axes[idx].bar_label(container, fmt="%.3f", fontsize=8)

plt.suptitle("Metricas de evaluacion en Test Set", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# matrices de confusion
n_models = len(confusion_matrices)
cols = 4
rows = (n_models + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
axes = axes.flatten()

for idx, (name, cm) in enumerate(confusion_matrices.items()):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No Churn", "Churn"],
                yticklabels=["No Churn", "Churn"],
                ax=axes[idx])
    axes[idx].set_title(name, fontsize=10)
    axes[idx].set_ylabel("Real")
    axes[idx].set_xlabel("Prediccion")

for idx in range(n_models, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle("Matrices de Confusion", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# curvas ROC
plt.figure(figsize=(10, 8))

for name, model in trained_models.items():
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_proba = model.decision_function(X_test)
    else:
        continue

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random (AUC = 0.500)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curvas ROC - Modelos de Churn")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 7. Interpretacion de Resultados

### 7.1 Feature Importance

Analizamos que variables son mas importantes para la prediccion de churn.

In [ ]:
importance_data = get_feature_importance(trained_models, feature_names)

for name, importance in importance_data.items():
    top = importance.head(15)
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(x=top.values, y=top.index, orient="h", ax=ax, color="steelblue")
    ax.set_title(f"Top 15 Features - {name}")
    ax.set_xlabel("Importancia")
    plt.tight_layout()
    plt.show()

### 7.2 Classification Report detallado

In [ ]:
for name, report in reports.items():
    print(f"{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    report_df = pd.DataFrame(report).T
    print(report_df.round(4))
    print()

### 7.3 Interpretacion por modelo

**Logistic Regression:** Modelo lineal que asigna un peso a cada feature. Los coeficientes indican la direccion e intensidad de la relacion con el churn. Es interpretable y sirve como baseline solido.

**Decision Tree:** Modelo basado en reglas que divide los datos segun umbrales en las features. Propenso al sobreajuste si no se controla la profundidad.

**Random Forest:** Ensemble de arboles de decision. Mas robusto que un arbol individual, menos propenso al sobreajuste. Feature importance basada en la reduccion de impureza promedio.

**Gradient Boosting:** Construye arboles secuencialmente, cada uno corrigiendo errores del anterior. Generalmente el modelo con mejor rendimiento, pero mas costoso computacionalmente.

**SVM:** Busca el hiperplano optimo que separa las clases. Efectivo en espacios de alta dimension. Menos interpretable que los modelos basados en arboles.

**KNN:** Clasifica basandose en los k vecinos mas cercanos. Sensible a la escala de las features (por eso el escalado previo es importante).

**Naive Bayes:** Asume independencia entre features. Rapido y funciona bien con features categorizadas. Puede ser ingenuo en datasets con features correlacionadas.

## 8. Reduccion de Dimensionalidad con PCA

Aplicamos PCA (Principal Component Analysis) para:
1. Reducir la dimensionalidad del dataset
2. Evaluar cuantos componentes principales capturan la mayor parte de la varianza
3. Comparar el rendimiento de los modelos con y sin PCA
4. Visualizar los datos en 2D

In [ ]:
# analisis de varianza explicada con todos los componentes
pca_full = PCA(random_state=42)
pca_full.fit(features)

cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_, color="steelblue", alpha=0.7)
axes[0].set_title("Varianza explicada por componente")
axes[0].set_xlabel("Componente Principal")
axes[0].set_ylabel("Ratio de varianza explicada")

axes[1].plot(range(1, len(cumulative_variance) + 1), cumulative_variance, "bo-", markersize=4)
axes[1].axhline(y=0.95, color="r", linestyle="--", label="95% varianza")
axes[1].axhline(y=0.90, color="orange", linestyle="--", label="90% varianza")
axes[1].set_title("Varianza explicada acumulada")
axes[1].set_xlabel("Numero de componentes")
axes[1].set_ylabel("Varianza acumulada")
axes[1].legend()

plt.tight_layout()
plt.show()

n_95 = int(np.argmax(cumulative_variance >= 0.95) + 1)
n_90 = int(np.argmax(cumulative_variance >= 0.90) + 1)
print(f"Componentes para 90% de varianza: {n_90}")
print(f"Componentes para 95% de varianza: {n_95}")
print(f"Total de features originales: {features.shape[1]}")

In [ ]:
# aplicar PCA al train/test evitando data leakage (fit solo en train)
pca_model = PCA(n_components=n_95, random_state=42)
X_train_pca = pd.DataFrame(
    pca_model.fit_transform(X_train),
    columns=[f"PC{i+1}" for i in range(n_95)],
    index=X_train.index
)
X_test_pca = pd.DataFrame(
    pca_model.transform(X_test),
    columns=[f"PC{i+1}" for i in range(n_95)],
    index=X_test.index
)
print(f"Features reducidas: {X_train.shape[1]} -> {X_train_pca.shape[1]}")

# entrenar modelos con PCA
print("\nEntrenando modelos con PCA...")
trained_models_pca, cv_results_pca = train_and_validate(X_train_pca, y_train, save_models=False)

# evaluar con PCA
metrics_pca, _, _, _, _ = evaluate_all_models(
    trained_models_pca, X_test_pca, y_test,
    output_csv=os.path.join(RESULTS_DIR, "churn_metrics_pca.csv")
)

In [ ]:
# comparacion con y sin PCA
comparison = pd.DataFrame({
    "F1 sin PCA": metrics_df["f1"],
    "F1 con PCA": metrics_pca["f1"],
    "Diferencia": metrics_pca["f1"] - metrics_df["f1"]
}).round(4)

print("Comparacion de F1 Score: sin PCA vs con PCA")
print(comparison)

fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(comparison))
width = 0.35
ax.bar([i - width/2 for i in x], comparison["F1 sin PCA"], width, label="Sin PCA", color="steelblue")
ax.bar([i + width/2 for i in x], comparison["F1 con PCA"], width, label=f"Con PCA ({n_95} comp.)", color="coral")
ax.set_xticks(list(x))
ax.set_xticklabels(comparison.index, rotation=45, ha="right")
ax.set_ylabel("F1 Score")
ax.set_title("Comparacion de rendimiento: Sin PCA vs Con PCA")
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

In [ ]:
# visualizacion PCA 2D
pca_2d = PCA(n_components=2, random_state=42)
features_2d = pca_2d.fit_transform(features)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(features_2d[:, 0], features_2d[:, 1],
                     c=target, cmap="RdYlGn_r", alpha=0.4, s=10)
plt.colorbar(scatter, label="Churn")
ax.set_xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} varianza)")
ax.set_ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} varianza)")
ax.set_title("Proyeccion PCA 2D - Churn vs No Churn")
plt.tight_layout()
plt.show()

## 9. Segmentacion de Clientes con K-Means

Usamos K-Means (aprendizaje no supervisado) para segmentar clientes en grupos
y analizar como se relacionan los segmentos con la tasa de churn.

In [ ]:
# metodo del codo y silhouette score
inertias = []
silhouettes = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(features)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(features, labels, sample_size=5000, random_state=42))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), inertias, "bo-")
axes[0].set_title("Metodo del Codo (Elbow)")
axes[0].set_xlabel("Numero de clusters (k)")
axes[0].set_ylabel("Inercia")

axes[1].plot(list(k_range), silhouettes, "ro-")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("Numero de clusters (k)")
axes[1].set_ylabel("Silhouette Score")

plt.tight_layout()
plt.show()

optimal_k = list(k_range)[np.argmax(silhouettes)]
print(f"K optimo segun Silhouette Score: {optimal_k}")

In [ ]:
# aplicar K-Means con k optimo
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(features)

# agregar clusters al dataset original
churn_with_clusters = churn_data.copy()
churn_with_clusters["cluster"] = cluster_labels

# analisis por cluster
cluster_analysis = churn_with_clusters.groupby("cluster").agg(
    total_customers=("churn", "count"),
    churn_rate=("churn", "mean"),
    avg_recency=("recency_days", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary_total", "mean"),
    avg_review=("avg_review_score", "mean")
).round(3)

print("Analisis de segmentos de clientes:")
cluster_analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# clusters en PCA 2D
scatter1 = axes[0].scatter(features_2d[:, 0], features_2d[:, 1],
                           c=cluster_labels, cmap="viridis", alpha=0.4, s=10)
plt.colorbar(scatter1, ax=axes[0], label="Cluster")
axes[0].set_title("Segmentos K-Means (PCA 2D)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

# tasa de churn por cluster
cluster_churn = churn_with_clusters.groupby("cluster")["churn"].mean()
axes[1].bar(cluster_churn.index, cluster_churn.values, color="coral")
axes[1].set_title("Tasa de Churn por Segmento")
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Tasa de Churn")
for i, v in enumerate(cluster_churn.values):
    axes[1].text(i, v + 0.01, f"{v:.1%}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

## 10. Conclusiones

### Resumen de resultados

**Modelos evaluados:** Se entrenaron y evaluaron 7 modelos de clasificacion para predecir churn de clientes.
Cada modelo fue validado con StratifiedKFold (5 folds) y evaluado en un test set independiente (20% de los datos).

**Control de sobreajuste:** Se comparo el rendimiento en entrenamiento vs validacion cruzada para cada modelo.
Los modelos con mayor gap entre train y CV muestran tendencia al sobreajuste.

**PCA:** La reduccion de dimensionalidad con PCA permite reducir el numero de features manteniendo el 95%
de la varianza. El impacto en el rendimiento depende del modelo y la complejidad del dataset.

**K-Means:** La segmentacion de clientes revela grupos con diferentes tasas de churn, lo que permite
al equipo de retencion priorizar acciones en los segmentos de mayor riesgo.

**Modelos y tecnicas cubiertos en este notebook:**
- Regresion Logistica
- Arbol de Decision
- Random Forest
- Gradient Boosting
- SVM
- k-NN
- Naive Bayes
- PCA (reduccion de dimensionalidad)
- K-Means (segmentacion de clientes)

In [ ]:
print("RESUMEN FINAL")
print("=" * 60)
print(f"\nMejor modelo por F1 Score: {metrics_df['f1'].idxmax()} ({metrics_df['f1'].max():.4f})")
print(f"Mejor modelo por Accuracy: {metrics_df['accuracy'].idxmax()} ({metrics_df['accuracy'].max():.4f})")
print(f"Mejor modelo por Recall:   {metrics_df['recall'].idxmax()} ({metrics_df['recall'].max():.4f})")
print(f"\nTotal de clientes analizados: {len(churn_data)}")
print(f"Tasa de churn: {churn_data['churn'].mean():.2%}")
print(f"Features utilizadas: {len(feature_names)}")
print(f"Componentes PCA (95% varianza): {n_95}")
print(f"Segmentos K-Means: {optimal_k}")